In [1]:
import torch
import torch.nn as nn
import torchvision.datasets as dataserts
import torchvision.transforms as transforms
from torch.autograd import Variable

In [2]:
input_size= 784       # 28*28
hidden_size= 400
out_size= 10          # 0-9
epochs= 10
batch_Size = 100
learning_rate= 0.001

In [3]:
train_dataset =dataserts.MNIST(root='./data',train=True,transform=transforms.ToTensor(),download=True)
test_dataset =dataserts.MNIST(root='./data',train=False,transform=transforms.ToTensor())

100%|██████████| 9.91M/9.91M [00:00<00:00, 18.5MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 498kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.58MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 13.0MB/s]


In [4]:
train_loader = torch.utils.data.DataLoader(dataset=train_dataset,batch_size=batch_Size,shuffle=True)
test_loader = torch.utils.data.DataLoader(dataset=test_dataset,batch_size=batch_Size,shuffle=False)

In [5]:
class Net(nn.Module): # Model, Network, NeuralNetwork, Net, CNN, RNN...
    def __init__(self, input_size, hidden_size, out_size):
        super(Net, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.fc3 = nn.Linear(hidden_size, out_size)
        self.relu = nn.ReLU() #activation Function

    def forward(self, x):   #forward propagation
        out = self.fc1(x)       #input
        out = self.relu(out)    #activation
        out = self.fc2(out)     #hidden
        out = self.relu(out)    #activation
        out = self.fc3(out)     #output
        return out

In [6]:
model = Net(input_size, hidden_size, out_size)

In [8]:
CUDA = torch.cuda.is_available()
if CUDA:
    model = model.cuda()

In [9]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [13]:
list(model.parameters())

[Parameter containing:
 tensor([[ 0.0101, -0.0107, -0.0235,  ...,  0.0192,  0.0313, -0.0321],
         [ 0.0183,  0.0200,  0.0080,  ...,  0.0048,  0.0117,  0.0045],
         [-0.0041,  0.0165,  0.0089,  ..., -0.0301, -0.0204,  0.0177],
         ...,
         [ 0.0343,  0.0326, -0.0171,  ...,  0.0085, -0.0096,  0.0328],
         [ 0.0077, -0.0016,  0.0300,  ..., -0.0305, -0.0184, -0.0159],
         [-0.0183, -0.0189,  0.0316,  ..., -0.0330, -0.0241,  0.0303]],
        device='cuda:0', requires_grad=True),
 Parameter containing:
 tensor([ 0.0289, -0.0052, -0.0326, -0.0211, -0.0203,  0.0076,  0.0078, -0.0277,
         -0.0190, -0.0216, -0.0218,  0.0058, -0.0260,  0.0314, -0.0311,  0.0121,
         -0.0109,  0.0127,  0.0064, -0.0218,  0.0251, -0.0047,  0.0239,  0.0298,
          0.0093, -0.0004,  0.0302,  0.0273,  0.0046, -0.0082,  0.0210,  0.0035,
         -0.0174, -0.0254, -0.0080, -0.0184,  0.0163, -0.0347, -0.0205,  0.0293,
          0.0310,  0.0253,  0.0300,  0.0092,  0.0338, -0.0216,

In [14]:
model.parameters

<bound method Module.parameters of Net(
  (fc1): Linear(in_features=784, out_features=400, bias=True)
  (fc2): Linear(in_features=400, out_features=400, bias=True)
  (fc3): Linear(in_features=400, out_features=10, bias=True)
  (relu): ReLU()
)>

In [15]:
for images, labels in train_loader:
    print(images.size())
    images = images.view(-1, 28*28)
    print(images.size())
    break

torch.Size([100, 1, 28, 28])
torch.Size([100, 784])


In [19]:
total_train = 0
correct_train = 0
for epoch in range(epochs):
    for i, (images, labels) in enumerate(train_loader):

        # Flatten the image
        images = Variable(images.view(-1, 28*28))
        labels = Variable(labels)

        if CUDA:
            images = images.cuda()
            labels = labels.cuda()

        # Clear the param_grad (Gradient'i sifirla)
        optimizer.zero_grad()
        outputs = model(images)                             # Forward Pass
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)

        if CUDA:
            correct_train += (predicted.cpu() == labels.cpu()).sum()
        else:
            correct_train += (predicted == labels).sum()

        loss = criterion(outputs, labels)                   # Difference between the actual and predicted (loss function)
        loss.backward()                                     # Backpropagation
        optimizer.step()                                    # Update the weights

        if (i+1) % 100 == 0:
            print('Epoch [{}/{}], Iteration: [{}/{}], Training Loss: {}%, Training Accuracy: {}%'
                  .format(epoch+1, epochs, i+1, len(train_dataset)//batch_Size, loss.item(), (100*correct_train/total_train)
                 ))

print("Training is done!")

Epoch [1/10], Iteration: [100/600], Training Loss: 0.2185586392879486%, Training Accuracy: 92.12999725341797%
Epoch [1/10], Iteration: [200/600], Training Loss: 0.12035588920116425%, Training Accuracy: 92.8949966430664%
Epoch [1/10], Iteration: [300/600], Training Loss: 0.10587882250547409%, Training Accuracy: 93.46666717529297%
Epoch [1/10], Iteration: [400/600], Training Loss: 0.10188503563404083%, Training Accuracy: 93.90249633789062%
Epoch [1/10], Iteration: [500/600], Training Loss: 0.15127216279506683%, Training Accuracy: 94.33599853515625%
Epoch [1/10], Iteration: [600/600], Training Loss: 0.052718326449394226%, Training Accuracy: 94.65666961669922%
Epoch [2/10], Iteration: [100/600], Training Loss: 0.11036042124032974%, Training Accuracy: 94.9914321899414%
Epoch [2/10], Iteration: [200/600], Training Loss: 0.03891737014055252%, Training Accuracy: 95.24749755859375%
Epoch [2/10], Iteration: [300/600], Training Loss: 0.07585065811872482%, Training Accuracy: 95.47333526611328%
Epo

In [20]:
# Test the Network (No loss and weight calculation, no weight update)

correct = 0
total = 0

for images, labels in test_loader:

    images = Variable(images.view(-1, 28*28))

    if CUDA:
        images = images.cuda()

    # For each input(sample/row) in the batch, the output will contain 10 elements.
    outputs = model(images)
    _, predicted = torch.max(outputs.data, 1)
    # predicted = outputs.data.max(1) [1]
    total += labels.size(0)

    # Also : correct += predicted.eq(labels).sum()
    if CUDA:
        correct += (predicted.cpu() == labels.cpu()).sum()
    else:
        correct += (predicted == labels).sum()

print('Final Test Accuracy: %d%%' % (100*correct/total))

Final Test Accuracy: 97%
